# 协整作用与相关性 (Cointegration vs. Correlation)

在探索多资产交易策略（如配对交易）时，理解“相关性”和“协整性”的区别绝对是最重要的一环。许多新手都会犯把高相关性等同于可配对的错误。

## 1. 相关性 (Correlation) 骗局

**相关性 (Pearson Correlation)** 衡量两组数据在**短期内（通常是日收益率）** 一起变动的程度。取值范围从 -1 到 1。

*   **+1**: 完美正相关。
*   **0**: 不相关。
*   **-1**: 完美负相关。

**陷阱：高相关并不意味着它们长期走势在一起。**

假设两支股票：
*   股票A：每天以 5% 的概率随机涨跌1美元，但有一条轻微向上的长期趋势线。
*   股票B：每天与股票A同涨同跌（相关性近乎为1），但它有一条更陡峭的长期向上趋势。

即使它们每天的收益率相关性很高，一年后它们的差价（Spread，即 A 价格减去 B 价格）会越拉越大，永远不会回到初始均值。**基于高相关性做均值回归配对交易（做空涨得多，做多涨得少），将会破产。**

## 2. 协整性 (Cointegration) - 真正的灵魂伴侣

**协整性** 考量的是长期（价格本身，而不是每日收益率）平衡关系。

如果两个时间序列 $X_t$ 和 $Y_t$ 分别都是非平稳的（各自随机游走），但存在某个线性组合：
$$ S_t = Y_t - \beta X_t $$
使得 $S_t$ 是**平稳的**（均值回归的），那么我们称 $X_t$ 和 $Y_t$ 是**协整的**。

*   这里的 $S_t$ 是它们之间的**差价 (Spread)**。
*   $\beta$ 是**对冲比例 (Hedge Ratio)**。

**直观的比喻（醉汉与狗）：**
*   **相关性**：两个互不认识的醉汉在广场上走，恰好连续三次都向右迈步。这叫相关。
*   **协整性**：一个醉汉牵着他的狗。醉汉路线随机，狗路线也随机。但由于有狗绳牵着，它们之间的距离（组合）是有上限的（平稳的）。当距离拉大到狗绳极限，一方必定会被拉回来。配对交易就是在寻找这根“狗绳”。

## 3. Engle-Granger 协整检验

通常使用两步法测试协整：
1.  **普通最小二乘法 (OLS) 回归：** 用 $X_t$ 回归 $Y_t$，得到回归系数 $\beta$，其残差序列为差价 $S_t$。
2.  **ADF 平稳性检验：** 对残差序列 $S_t$ 进行 ADF 检验。如果它平稳，说明系统具备协整关系。


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.tsa.stattools as ts
import matplotlib.pyplot as plt

# --- 模拟两只高度相关的股票，但它们不协整 (分别带有各自的漂移趋势) ---
np.random.seed(42)
returns_base = np.random.normal(0, 1, 500)
price_A = np.cumsum(returns_base + 0.1)  # 漂移项较小
price_B = np.cumsum(returns_base + 0.25) # 漂移项更大

plt.figure(figsize=(12, 5))
plt.plot(price_A, label='Stock A (Drift = 0.1)')
plt.plot(price_B, label='Stock B (Drift = 0.25)')
plt.title('High Correlation, No Cointegration (Spurious correlation)')
plt.legend()
plt.show()

# 它们的相关性非常高
pca_corr = np.corrcoef(np.diff(price_A), np.diff(price_B))[0, 1]
print(f'日收益率相关性: {pca_corr:.4f} (极高相关性)')

# 协整检验
_, pval_coint, _ = ts.coint(price_A, price_B)
print(f'协整检验 p-value: {pval_coint:.4f} (大于0.05，拒绝协整)')


## 4. 统计套利的基础

找到了协整关系，我们就构建了**统计套利 (Statistical Arbitrage)** 中最经典的**配对交易 (Pairs Trading)** 的基础：
1. 计算差价序列。
2. 将差价标准化为**Z-Score**。
3. 当 Z-Score > 2 时，做空差价组合（做空 Y，做多 $\beta$ 份 X）。
4. 当 Z-Score < -2 时，做多差价组合（做多 Y，做空 $\beta$ 份 X）。
5. 当差价回归均值 (Z-Score=0) 时平仓获利。

在下一章中我们会完整实现这个策略代码。


## 5. 实战：在股票池中寻找真协整资产对

现实中，量化研究员通常会圈定一个行业的股票池（比如银行股、航空股），然后在这个池子里两两计算 Engle-Granger 协整检验。找出 P-value 小于 0.05 甚至 0.01 的配对。

这往往由一个**热力图 (Heatmap)** 来实现，这是一种极其直观的可视化方法。


In [ ]:
import seaborn as sns
import numpy as np
import pandas as pd
import statsmodels.tsa.stattools as ts
import matplotlib.pyplot as plt

# --- 模拟构建一个包含 5 只股票的股票池 ---
np.random.seed(10)
returns_pool = np.random.normal(0, 1, (500, 5))
base_prices = np.cumsum(returns_pool, axis=0) + 100
pool_df = pd.DataFrame(base_prices, columns=['Stock_A', 'Stock_B', 'Stock_C', 'Stock_D', 'Stock_E'])

# 特意制造一对协整关系 (让 Stock_E 成为 Stock_B 的协整影子)
pool_df['Stock_E'] = pool_df['Stock_B'] * 1.5 - 20 + np.random.normal(0, 2, 500)

# 计算协整 P-value 矩阵
def find_cointegrated_pairs(dataframe):
    n = dataframe.shape[1]
    pvalue_matrix = np.ones((n, n))
    keys = dataframe.columns
    pairs = []
    for i in range(n):
        for j in range(i+1, n):
            S1 = dataframe[keys[i]]
            S2 = dataframe[keys[j]]
            result = ts.coint(S1, S2)
            pvalue = result[1]
            pvalue_matrix[i, j] = pvalue
            if pvalue < 0.05:
                pairs.append((keys[i], keys[j], pvalue))
    return pvalue_matrix, pairs

pvalues, cointegrated_pairs = find_cointegrated_pairs(pool_df)

print("发现的协整配对:")
for pair in cointegrated_pairs:
    print(f"{pair[0]} 和 {pair[1]} - P-value: {pair[2]:.4f}")

# 绘制热力图 P-value Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(pvalues, xticklabels=pool_df.columns, yticklabels=pool_df.columns, cmap='viridis_r', 
            mask=(pvalues >= 0.99), annot=True)
plt.title('Cointegration P-values Heatmap')
plt.show()


## 🎯 练习

1. 对 A 股银行板块（例如工商银行、建设银行、农业银行）的股票做协整检验，找出配对。
2. 画出任意两只高相关股票 5 年内的价格走势图，观察它们的价差是否保持平稳或者最终发散。
3. 修改 `find_cointegrated_pairs` 函数，加入参数让用户自定义置信水平（0.05 或 0.01）。

---
**下一节** → `../04_backtesting/06_pairs_trading_strategy.ipynb`
